In [2]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.getcwd())

from src.models import ResNetFrozen
from src.train import train
from src.dataset import HAM10000Dataset, get_train_transforms, get_eval_transforms, compute_class_weights, IMAGENET_MEAN, IMAGENET_STD, DX_TO_IDX, IDX_TO_DX
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

Read in training and validation data, initialize HAM10000Dataset objects for each

In [3]:
train_df = pd.read_csv('data/splits/train.csv')
val_df = pd.read_csv('data/splits/val.csv')

image_dir = os.path.join(os.getcwd(), 'data', 'HAM10000_images')

train_dataset = HAM10000Dataset(train_df, image_dir, get_train_transforms())
val_dataset = HAM10000Dataset(val_df, image_dir, get_eval_transforms())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)

Create model from ResNet-18 pre-trained model with a frozen backbone, with just a final classifier linear layer unfrozen 

In [4]:
model = ResNetFrozen()

Construct Loss Function

In [5]:
class_weights = compute_class_weights(train_df)

loss_func = torch.nn.CrossEntropyLoss(weight=class_weights)

Construct Optimizer

In [6]:
lr = 1e-3
momentum = 0.9
adaptive_learning_rate = 0.999
betas = (momentum, adaptive_learning_rate)
optim_params = (p for p in model.parameters() if p.requires_grad)
optimizer = torch.optim.AdamW(optim_params, lr=lr, betas=betas)

Train model

In [8]:
model = train(
    model=model,
    model_name="ResNetFrozen",
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=loss_func,
    optimizer=optimizer,
    num_epoch=100
)

Using CUDA
Epoch  1 / 100: train_acc=0.6520 val_acc=0.6934 train_f1=0.4957 val_f1=0.5246 
Epoch 10 / 100: train_acc=0.6645 val_acc=0.6751 train_f1=0.5131 val_f1=0.5144 
Epoch 20 / 100: train_acc=0.6522 val_acc=0.6785 train_f1=0.4972 val_f1=0.4753 
Epoch 30 / 100: train_acc=0.6584 val_acc=0.6216 train_f1=0.5040 val_f1=0.4693 
Epoch 40 / 100: train_acc=0.6684 val_acc=0.6316 train_f1=0.5064 val_f1=0.4849 
Epoch 50 / 100: train_acc=0.6649 val_acc=0.6791 train_f1=0.5164 val_f1=0.5014 
Epoch 60 / 100: train_acc=0.6623 val_acc=0.6590 train_f1=0.5170 val_f1=0.4987 
Epoch 70 / 100: train_acc=0.6618 val_acc=0.6151 train_f1=0.5126 val_f1=0.4558 
Epoch 80 / 100: train_acc=0.6526 val_acc=0.7320 train_f1=0.5061 val_f1=0.5336 
Epoch 90 / 100: train_acc=0.6703 val_acc=0.6953 train_f1=0.5339 val_f1=0.5055 
Epoch 100 / 100: train_acc=0.6563 val_acc=0.5992 train_f1=0.5067 val_f1=0.4521 
Model saved to logs\ResNetFrozen_0707_212302\ResNetFrozen.th
